# DP-MLM project · LLM-based evaluation (runs on the saved outputs)
Second notebook of the project. It does **not** rerun DP-MLM: it reads the rewrites saved by
`dp_mlm_replication.ipynb` (`results.zip`) and adds

1. environment details and the calibrated clipping bounds of the `dpmlm` package,
2. BLEU / SBERT cosine for the privatised Reddit test posts,
3. **LLM-as-judge** ratings (meaning, fluency) with identical / mismatched control pairs,
4. the LLM as a **second annotator** of the constructed labels + **Cohen's κ** (and against the human audit if provided),
5. a **zero-shot LLM attacker** for gender and age bracket.

**Run:** T4 GPU → run the first cell and upload `results.zip` (and optionally `reddit_audit_human.csv`) → `Runtime → Run all`. About 15–20 minutes. Downloads `llm_results.zip`.

In [ ]:
# --- Upload results.zip from the first notebook (and optionally reddit_audit_human.csv) ---
import os, zipfile
try:
    from google.colab import files
    up = files.upload()
except Exception:
    up = {}
if os.path.exists("results.zip"):
    zipfile.ZipFile("results.zip").extractall(".")
if os.path.exists("reddit_audit_human.csv"):
    os.replace("reddit_audit_human.csv", "results/reddit_audit_human.csv")
print(sorted(os.listdir("results")))

In [ ]:
!pip install -q dpmlm sentence-transformers sacrebleu accelerate
import nltk
for p in ["stopwords", "wordnet", "punkt", "punkt_tab", "words", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(p, quiet=True)
import importlib.util, pathlib
core = pathlib.Path(importlib.util.find_spec("dpmlm").origin).parent / "core.py"
src = core.read_text(); line = "torch.backends.cuda.matmul.fp32_precision = 'tf32'"
if line in src and "fp32_patch" not in src:
    core.write_text(src.replace(line, "try:  # fp32_patch\n    " + line + "\nexcept Exception:\n    pass"))

In [ ]:
import re, json, platform, importlib.metadata as im
import numpy as np, pandas as pd, torch, transformers, sentence_transformers, sacrebleu
from tqdm.auto import tqdm
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score
SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)
RES = "results"; OUT = "results/llm"; os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPSILONS = [10, 25, 50, 100, 250]

# environment + clipping bounds of the package used for all rewrites
from dpmlm import DPMLM
M = DPMLM()
JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
env = {"EnvGPU": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU", "EnvTorch": torch.__version__,
       "EnvTransformers": transformers.__version__, "EnvPython": platform.python_version(), "EnvDpmlm": im.version("dpmlm"),
       "EnvSbert": sentence_transformers.__version__, "EnvSacrebleu": sacrebleu.__version__,
       "ClipMin": f"{M.clip_min:.2f}", "ClipMax": f"{M.clip_max:.2f}", "Sensitivity": f"{M.sensitivity:.2f}", "JudgeModel": JUDGE_MODEL}
json.dump(env, open(f"{OUT}/env.json", "w"), indent=1); print(env)
del M; torch.cuda.empty_cache()

## 1 · BLEU and cosine similarity of the Reddit rewrites

In [ ]:
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
def cos_each(a, b):
    ea = sbert.encode(list(a), normalize_embeddings=True, batch_size=64); eb = sbert.encode(list(b), normalize_embeddings=True, batch_size=64)
    return (ea * eb).sum(1)
tp = pd.read_csv(f"{RES}/reddit_test_privatized.csv"); tp["privatized"] = tp["privatized"].astype(str)
rq = pd.DataFrame([{"epsilon": e, "BLEU": sacrebleu.corpus_bleu(list(g.privatized), [list(g.original)]).score,
                    "CosSim": float(cos_each(g.original, g.privatized).mean())} for e, g in tp.groupby("epsilon")]).round(3)
rq.to_csv(f"{OUT}/reddit_quality.csv", index=False); display(rq)

## 2 · Load the judge LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
jtok = AutoTokenizer.from_pretrained(JUDGE_MODEL, padding_side="left")
jlm = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
                                           device_map="auto" if DEVICE == "cuda" else None).eval()
def llm_generate(prompts, system, max_new_tokens=16, batch_size=16):
    outs = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="LLM", leave=False):
        chats = [jtok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": p}],
                                          tokenize=False, add_generation_prompt=True) for p in prompts[i:i + batch_size]]
        enc = jtok(chats, return_tensors="pt", padding=True).to(jlm.device)
        with torch.no_grad():
            gen = jlm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=jtok.eos_token_id)
        outs += jtok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

## 3 · LLM-as-judge: meaning preservation and fluency (1–5)

In [ ]:
JUDGE_SYSTEM = "You are a careful annotator who evaluates rewritten English text. Answer only in the requested format."
JUDGE_TMPL = """Text A is an original text. Text B is an automatically rewritten version of it.

Text A: {a}
Text B: {b}

Rate Text B on two 1-5 scales.
Meaning: how well does B preserve the meaning of A? (1 = unrelated, 3 = partly preserved, 5 = same meaning)
Fluency: how natural and grammatical is B as English, ignoring A? (1 = gibberish, 3 = understandable but awkward, 5 = fully fluent)
Answer exactly as: Meaning: <1-5>; Fluency: <1-5>"""
def parse_judge(s):
    m = re.search(r"Meaning:\s*([1-5]).*?Fluency:\s*([1-5])", s, re.S | re.I)
    return (int(m[1]), int(m[2])) if m else (np.nan, np.nan)

N_JUDGE = 40; JUDGE_EPS = [10, 50, 250]
sp = pd.read_csv(f"{RES}/sst2_privatized.csv"); sp["privatized"] = sp["privatized"].astype(str)
sst_ids = sorted(sp.id.unique())[:N_JUDGE]
red_orig = list(tp[tp.epsilon == 10].original)[:N_JUDGE]
pools = {"SST-2": ([sp[(sp.id == i)].original.iloc[0] for i in sst_ids],
                   {e: [sp[(sp.id == i) & (sp.epsilon == e)].privatized.iloc[0] for i in sst_ids] for e in JUDGE_EPS}),
         "Reddit": (red_orig, {e: [tp[(tp.original == o) & (tp.epsilon == e)].privatized.iloc[0] for o in red_orig] for e in JUDGE_EPS})}
rng = np.random.RandomState(SEED); rows = []
for ds, (orig, priv) in pools.items():
    for i, o in enumerate(orig):
        rows.append({"dataset": ds, "condition": "identical (control)", "id": i, "a": o, "b": o})
        j = (i + 1 + rng.randint(len(orig) - 1)) % len(orig)
        rows.append({"dataset": ds, "condition": "mismatched (control)", "id": i, "a": o, "b": orig[j]})
        for e in JUDGE_EPS: rows.append({"dataset": ds, "condition": f"eps={e}", "id": i, "a": o, "b": priv[e][i]})
judge = pd.DataFrame(rows)
judge["raw"] = llm_generate([JUDGE_TMPL.format(a=a, b=b) for a, b in zip(judge.a, judge.b)], JUDGE_SYSTEM)
judge[["meaning", "fluency"]] = [parse_judge(r) for r in judge.raw]
judge["cossim"] = cos_each(judge.a, judge.b)
judge.to_csv(f"{OUT}/judge_scores.csv", index=False)
summ = judge.groupby(["dataset", "condition"])[["meaning", "fluency", "cossim"]].mean().round(2).reset_index()
summ.to_csv(f"{OUT}/judge_summary.csv", index=False)
rw = judge[judge.condition.str.startswith("eps")].dropna(subset=["meaning"])
rho, p = spearmanr(rw.meaning, rw.cossim)
stats = pd.DataFrame({"stat": ["NJudge", "JudgeParse", "JudgeRho", "JudgeRhoN", "JudgeRhoP"],
                      "value": [N_JUDGE, f"{100 * judge.meaning.notna().mean():.1f}", f"{rho:.2f}", len(rw), f"{p:.1e}"]})
stats.to_csv(f"{OUT}/judge_stats.csv", index=False)
display(summ); display(stats)

## 4 · LLM as second annotator of the constructed labels (Cohen's κ)
Human audit (optional): fill `human_gender` (M/F/?) and `human_age_bracket` (<=25/>25/?) in `reddit_audit_human.csv`
(it does not show the regex labels) and upload it together with `results.zip` in the first cell.

In [ ]:
ANN_SYSTEM = "You extract information that a Reddit poster states about themselves. Answer only in the requested format."
ANN_TMPL = """Reddit title: {t}
Reddit post: {p}

What gender and age does the POSTER (the person who wrote this, not their partner or anyone else) state for themselves?
Answer exactly as: Gender: <M or F or ?>; Age: <number or ?>"""
def parse_ann(s):
    g = re.search(r"Gender:\s*([MF?])", s, re.I); a = re.search(r"Age:\s*(\d{1,2})", s)
    return (g[1].upper() if g and g[1] != "?" else "?"), (("<=25" if int(a[1]) <= 25 else ">25") if a else "?")
key = pd.read_csv(f"{RES}/reddit_audit_sample_100.csv").reset_index(drop=True)
key["audit_id"] = range(len(key))
raw = llm_generate([ANN_TMPL.format(t=t, p=str(p)[:1500]) for t, p in zip(key.title, key.post)], ANN_SYSTEM)
key[["llm_gender", "llm_age_bracket"]] = [parse_ann(r) for r in raw]; key["llm_raw"] = raw
def kap(a, b): a, b = [str(x).strip() for x in a], [str(x).strip() for x in b]; return cohen_kappa_score(a, b), float(np.mean(np.array(a) == np.array(b)))
ag = []
for attr, reg, llm in [("gender", "gender", "llm_gender"), ("age bracket", "age_bracket", "llm_age_bracket")]:
    k, r = kap(key[reg], key[llm]); ag.append({"attribute": attr, "pair": "regex vs LLM", "n": len(key), "kappa": k, "raw agreement": r})
hp = f"{RES}/reddit_audit_human.csv"
if os.path.exists(hp):
    h = pd.read_csv(hp).merge(key, on="audit_id")
    h = h[h.human_gender.astype(str).str.strip().isin(["M", "F", "?"])]
    for attr, reg, llm, hum in [("gender", "gender", "llm_gender", "human_gender"), ("age bracket", "age_bracket", "llm_age_bracket", "human_age_bracket")]:
        for name, col in [("regex vs human", reg), ("LLM vs human", llm)]:
            k, r = kap(h[col], h[hum]); ag.append({"attribute": attr, "pair": name, "n": len(h), "kappa": k, "raw agreement": r})
else:
    print("No human audit file - only regex vs LLM.")
ag = pd.DataFrame(ag).round(3); ag.to_csv(f"{OUT}/label_agreement.csv", index=False)
key.to_csv(f"{OUT}/audit_llm_labels.csv", index=False); display(ag)
display(key[(key.gender != key.llm_gender) | (key.age_bracket != key.llm_age_bracket)][["title", "age", "gender", "llm_raw"]].head(15))

## 5 · Zero-shot LLM attacker on the Reddit test posts

In [ ]:
ATT_SYSTEM = "You are a profiling model. Always give your best guess, never refuse. Answer only in the requested format."
ATT_TMPL = """Here is a Reddit post with all explicit age and gender tags removed:

{p}

Guess the author's gender and whether the author is 25 or younger.
Answer exactly as: Gender: <M or F>; Age: <25 or younger / over 25>"""
def parse_att(s):
    g = re.search(r"Gender:\s*([MF])", s, re.I)
    young = re.search(r"25 or younger|younger|<=\s*25", s, re.I); old = re.search(r"over 25|older|>\s*25", s, re.I)
    return (g[1].upper() if g else "M"), ("<=25" if (young and not old) else ">25")
red = pd.read_csv(f"{RES}/reddit_selfdisclosure_dataset.csv").drop_duplicates("text").set_index("text")
N_ATT = 100; orig = list(tp[tp.epsilon == 10].original)[:N_ATT]; lab = red.loc[orig]
conds = {"original": orig} | {e: [tp[(tp.original == o) & (tp.epsilon == e)].privatized.iloc[0] for o in orig] for e in EPSILONS}
res = []
for cond, texts in conds.items():
    preds = [parse_att(r) for r in llm_generate([ATT_TMPL.format(p=t) for t in texts], ATT_SYSTEM, max_new_tokens=20)]
    for attr, col, k in [("gender", "gender", 0), ("age bracket", "age_bracket", 1)]:
        y = list(lab[col]); p = [x[k] for x in preds]
        res.append({"attribute": attr, "epsilon": cond, "F1": 100 * f1_score(y, p, average="macro"), "Acc": 100 * accuracy_score(y, p)})
llm_att = pd.DataFrame(res).round(2); llm_att.to_csv(f"{OUT}/reddit_llm_attacker.csv", index=False)
display(llm_att.pivot_table(index="attribute", columns="epsilon", values="F1", sort=False))

In [ ]:
with zipfile.ZipFile("llm_results.zip", "w") as z:
    for f in os.listdir(OUT): z.write(os.path.join(OUT, f))
try:
    files.download("llm_results.zip")
except Exception:
    print("llm_results.zip written")